In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 

from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
from tqdm.auto import tqdm
import librosa

# Make stimuli for 2024 mono same vs different sex distractor control experiment
___
## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-distractor same sex
- 1-distractor different sex
___
Will be using foregrounds and cues from manifest:
`/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl`

Stim will be saved at `/om/user/imgriff/datasets/human_distractor_sex_2024`    
Manifest has been copied to this directory as `full_stim_manifest_w_fnames.pdpkl` 

## Load and prep manifests

#### Get max n targets from set of foregrounds

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_sex_2024')


In [3]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_sex_2024')
stim_out_path.mkdir(parents=True, exist_ok=True)


manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl')

n_targets = manifest.shape[0]
print(f"{n_targets} targets")

manifest["sex_cond"] = manifest.gender == manifest.distractor_gender 
manifest["sex_cond"] = manifest['sex_cond'].apply(lambda x: "Same" if x == True else "Different")
print(manifest.gender.value_counts())
print(manifest.sex_cond.value_counts())
n_per_gender = manifest.gender.value_counts().values[0]

SR = 44100


new_manifest_out_name = stim_out_path / "full_stim_manifest_w_fnames.pdpkl"

1440 targets
female    720
male      720
Name: gender, dtype: int64
Same         720
Different    720
Name: sex_cond, dtype: int64


In [4]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))

# handle contractions 
word_dict['were'] = word_dict["we're"]
word_dict['dont'] = word_dict["don't"]
word_dict['didnt'] = word_dict["didn't"]
word_dict['doesnt'] = word_dict["doesn't"]



manifest['word_int'] = manifest['word'].map(word_dict)
manifest['distractor_word_int'] = manifest['distractor_word'].map(word_dict)

manifest.to_pickle(new_manifest_out_name)

In [5]:
import warnings
warnings.filterwarnings('ignore')

## Make and write audio for human experiment 

Will use same set of curated unique words as in SWC and Popham style experiments 

### Prep condition list

In [6]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

conditions = ['same',
              'different']

condition_pairs = list(itertools.product(conditions, snrs))
condition_pairs.append(('SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}
print(cond_ix_dict)
out_name = stim_out_path / "human_distractor_sex_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(cond_ix_dict, f)


{0: ('same', -9), 1: ('same', -6), 2: ('same', -3), 3: ('same', 0), 4: ('same', 3), 5: ('different', -9), 6: ('different', -6), 7: ('different', -3), 8: ('different', 0), 9: ('different', 3), 10: ('SILENCE', 'inf')}


## Make stimuli 

In [7]:

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factors

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

def rms_normalize_db(wav, dBSPL, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    new_rms = 20e-6 * np.power(10, dBSPL/20)
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor


In [8]:
manifest.columns

Index(['distractor_client_id', 'distractor_clip_dur_in_s',
       'distractor_clip_end_in_s', 'distractor_clip_start_in_s',
       'distractor_corpus', 'distractor_gender', 'distractor_gender_int',
       'distractor_split', 'distractor_split_int', 'distractor_sr',
       'distractor_src_fn', 'distractor_total_file_duration_in_s',
       'distractor_word', 'src_ix', 'client_id', 'clip_dur_in_s',
       'clip_end_in_s', 'clip_start_in_s', 'corpus', 'gender', 'gender_int',
       'split', 'split_int', 'sr', 'src_fn', 'total_file_duration_in_s',
       'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'cue_src_fn',
       'cue_clip_start_in_s', 'cue_clip_end_in_s', 'gender_cond_td',
       'cue_clip_dur_in_s', 'sex_cond', 'word_int', 'distractor_word_int'],
      dtype='object')

In [9]:
unique_words = sorted(manifest.word.unique())

In [10]:
np.random.seed(1) # set seed 

out_dir = stim_out_path / 'sounds'

SR = 44100
SPL = 60 # in dB SPL 
# timing in seconds 
trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
new_rms = 0.02 # is 60dB in  amplitude e.g 20e-6pa * 10**(60/20)
mixture_onset = int((isi + signal_dur) * SR) # in frames


for cond_id, (cond, snr) in cond_ix_dict.items():
    stim_out_dir = out_dir / f"condition_{cond_id:02}"
    stim_out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving stimuli to {stim_out_dir.as_posix()}")

    if cond == 'same':
        # sample 360 examples, 180 male 180 female with no target word overlap.
        female_same = manifest[(manifest.sex_cond == 'Same') & (manifest.gender == 'female')].sample(n=180, replace=False, random_state=cond_id).reset_index(drop=True)
        male_same = manifest[(manifest.sex_cond == 'Same') & (~manifest.word.isin(female_same.word.values)) & (manifest.gender == 'male')].sample(n=180, replace=False, random_state=cond_id).reset_index(drop=True)

    elif cond == 'different' or cond == "SILENCE":
        # sample 360 examples, 180 male 180 female with no target word overlap.
        female_same = manifest[(manifest.sex_cond == 'Different') & (manifest.gender == 'female')].sample(n=180, replace=False, random_state=cond_id).reset_index(drop=True)
        male_same = manifest[(manifest.sex_cond == 'Different') & (~manifest.word.isin(female_same.word.values)) & (manifest.gender == 'male')].sample(n=180, replace=False, random_state=cond_id).reset_index(drop=True)

    assert ~male_same.word.isin(female_same.word.values).all(), "Sampled same word more than once"
    cond_manifest = pd.concat([female_same, male_same], axis=0, ignore_index=True)
    # sort cond_manifest by word 
    cond_manifest = cond_manifest.sort_values("word").reset_index(drop=True)
    assert (cond_manifest.word == unique_words).all() , "Missing words in condition manifest"

    ## Write out the manifest for this condition
    cond_manifest_out_name = stim_out_dir / "condition_manifest.pdpkl"
    cond_manifest.to_pickle(cond_manifest_out_name)
    ## Write out trials
    fname_array = cond_manifest[["cue_src_fn", "src_fn", "distractor_src_fn"]].values

    for ix, (cue_fn, target_fn, distractor_fn) in enumerate(tqdm(fname_array, leave=False)):
        # init output signal 
        trial_signal = np.zeros((int(SR * trial_dur)),dtype=np.float32)
        # load already cut/resampled/windowed signals 
        cue_signal, cue_sr = librosa.load(cue_fn, sr=SR)
        target_signal, target_sr = librosa.load(target_fn, sr=SR)
        distractor_signal, dist_sr = librosa.load(distractor_fn, sr=SR)

        cue_signal = util_audio.set_dBSPL(cue_signal, SPL)
        target_signal = util_audio.set_dBSPL(target_signal, SPL)
        distractor_signal = util_audio.set_dBSPL(distractor_signal, SPL)

        if snr == 'inf':
            mixture = target_signal 
        else:
            mixture = util_audio.combine_signal_and_noise(target_signal, distractor_signal, snr)
        mixture = util_audio.ramp_hann(mixture, win_dur, samplerate=SR)
        mixture, mixture_scale_factor = rms_normalize_db(mixture, SPL)
        # set cue to same level as target post scaling
        cue_signal = util_audio.ramp_hann(cue_signal, win_dur, samplerate=SR)
        cue_signal = util_audio.set_dBSPL(cue_signal, SPL)
        cue_signal *= mixture_scale_factor

        # add parts to trial signal
        trial_signal[ : len(cue_signal)] += cue_signal  
        trial_signal[mixture_onset : ] += mixture
        trial_signal = util_audio.set_dBSPL(trial_signal, SPL)

        # save trial signal
        # f string with digint padded to 3 places
        out_name = stim_out_dir / f'{ix:03}.wav'
        sf.write(out_name, trial_signal, samplerate=SR)



Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_00


  0%|          | 0/360 [00:00<?, ?it/s]

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_01


  0%|          | 0/360 [00:00<?, ?it/s]

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_02


...

  0%|          | 0/360 [00:00<?, ?it/s]

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_09


  0%|          | 0/360 [00:00<?, ?it/s]

Saving stimuli to /om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_10


  0%|          | 0/360 [00:00<?, ?it/s]

In [12]:
## Merge condition all_manifests for later analysis 
all_manifests = []
for manifest_path in out_dir.rglob('condition_manifest.pdpkl'):
    df = pd.read_pickle(manifest_path)
    df['condition_dir'] = manifest_path.parent.name
    all_manifests.append(df)

final_manifest = pd.concat(all_manifests, axis=0, ignore_index=True)

# save 
final_manifest_out_name = stim_out_path / "full_expmt_stim_manifest.pdpkl"
final_manifest.to_pickle(final_manifest_out_name)




### Make target word key 


In [13]:
[all(final_manifest.word.unique() == all_manifests[i].word.values) for i in range(len(all_manifests))]


[True, True, True, True, True, True, True, True, True, True, True]

In [16]:
import json
import pickle 


exmpt_word_dict = {i:w for i,w in enumerate(sorted(final_manifest.word.unique()))}

# save as pickle
out_name = stim_out_path /  "human_distractor_sex_word_key.pkl"
with open(out_name, 'wb') as f:
    pickle.dump(exmpt_word_dict, f)



words = list(exmpt_word_dict.values())
# save as json 
out_name = stim_out_path /  "human_distractor_sex_word_key.js"
with open(out_name, 'w') as f:
    json.dump({"dictionary":words}, f)



In [17]:
len(exmpt_word_dict)

360

In [38]:
## Make word Key .js from pickle of word dict
import IPython.display as ipd


In [49]:
trial_ix = 10
print(f"word: {exmpt_word_dict[trial_ix]}")
ipd.Audio(f"/om/user/imgriff/datasets/human_distractor_sex_2024/sounds/condition_04/{trial_ix:03}.wav")

word: against


## Make sure stimuli generation worked 

In [35]:
paths = list(Path('/om/user/imgriff/datasets/human_distractor_sex_2024/sounds/').rglob('*/*.wav'))

In [32]:
import librosa

In [39]:
bad_paths = []
for path in paths:
    try:
        librosa.get_duration(filename=path)
    except Exception as e:
        print(e)
        bad_paths.append(path)

## Add transcripts to manifest

In [2]:
import pandas as pd 

In [7]:
manifests = pd.read_pickle('/om/user/imgriff/datasets/human_distractor_sex_2024/full_stim_manifest_w_fnames.pdpkl')


In [9]:
manifests.gender_cond_td.value_counts()

female_female    360
female_male      360
male_female      360
male_male        360
Name: gender_cond_td, dtype: int64

In [19]:
### Manifest with transcripts

parent_dir = Path('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/')
gt_manifest = pd.read_pickle(parent_dir / 'full_eval_trial_manifest_new_fnames_w_transcripts.pdpkl')

In [20]:
manifests.head()

,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,...,cue_client_id,cue_src_fn,cue_clip_start_in_s,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s,sex_cond,word_int,distractor_word_int,condition_dir
0,etoile,0.58,90.28,89.70,swc,female,0,NaN,0,44100,...,flyingtoaster,/om/user/imgriff/datasets/spatial_audio_pipeli...,2231.59,2232.40,female_female,0.81,Same,1,158,condition_01
1,pilot-dawg,0.35,277.23,276.88,swc,male,1,NaN,0,44100,...,the-voice-of-hassocks,/om/user/imgriff/datasets/spatial_audio_pipeli...,3974.41,3975.09,male_male,0.68,Same,3,672,condition_01
2,divinemeaninglessness,0.56,297.14,296.58,swc,male,1,NaN,0,44100,...,mangst,/om/user/imgriff/datasets/spatial_audio_pipeli...,1502.80,1503.39,male_male,0.59,Same,4,401,condition_01
3,laura-s,0.21,992.47,992.26,swc,female,0,NaN,0,44100,...,popularoutcast,/om/user/imgriff/datasets/spatial_audio_pipeli...,113.20,113.58,female_female,0.38,Same,5,479,condition_01
4,hassocks5489,0.31,662.04,661.73,swc,male,1,NaN,0,44100,...,s-whistler,/om/user/imgriff/datasets/spatial_audio_pipeli...,1001.45,1001.95,male_male,0.50,Same,7,728,condition_01


In [21]:
## Match rows from manifest to gt_manifest and add transcripts from gt_manifest to manifest

# get columns in manifest that are also in gt_manifest
common_cols = list(set(manifests.columns).intersection(gt_manifest.columns))

manifests = pd.merge(manifests, gt_manifest, on=common_cols, how='inner')

# write out 
# # save 
final_manifest_out_name = stim_out_path / "full_expmt_stim_manifest.pdpkl"
manifests.to_pickle(final_manifest_out_name)


